In [1]:
import spagraph as spg

sc_file = f"/home/maweicheng/ST_data/GSE243275/GSM7782698_SC.h5ad"
st_file = f"/home/maweicheng/ST_data/GSE243275/GSM7782699_ST.h5ad"
output_dir = f"/home/maweicheng/ST_data/GSE243275/evaluate"   

# marker_selection_method="l1"
# art = spg.vae(sc_file=sc_file, st_file=st_file, resolution=4, top_n_per_type=100, output_dir=output_dir, precomputed_marker_file=f"{output_dir}/final_genes.txt")
art = spg.vae(sc_file=sc_file, 
              st_file=st_file,
              top_n_per_type=100, 
              output_dir=output_dir,
              device = "cuda:1"
              )

res = spg.deconv(vae=art, 
                 output_dir=output_dir, 
                 k_cells_per_cluster=15, 
                 k_celltype=[20], 
                 scale_basis="all", 
                 batch_size=512,
                 save_reconstructed_genes=True,
                 device= "cuda:1"
                 ) 

/softwares/miniconda3/envs/dl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Stage 1: VAE Training
  Epochs:        300
  LR:            0.0005
  Batch Size:    512
  Latent Dim:    256
  Beta (KL):     0.1
  Lambda MMD:    0.03
  Resolution:    4.0
  Top N/Type:    100
  Dual Decoder:  True
  Seed:          42
  Save to Disk:  True

✅ 自动计算 library_size factor: 1.0028
   - SC HVG 总count: 18360758
   - ST HVG 总count: 18310190
   - 使用HVG基因数: 2000
Total: 2130 marker genes
   Number of clusters: 70
Epoch 1/300 | train_loss=1112.8140 | test_loss=797.7265
Epoch 50/300 | train_loss=654.8285 | test_loss=597.4346
Epoch 100/300 | train_loss=642.5062 | test_loss=587.8565
Early stopping at epoch 139/300, best_loss=586.4752
Stage 1 config saved to: /home/maweicheng/ST_data/GSE243275/evaluate/config_vae.txt
✅ 使用 Stage1 自动计算的 library_size: 1.0028

Stage 2: GAT Deconvolution
Sample:             GSM7782699_ST
Epochs:             300
LR:                 0.005
Batch Size:         512
K Spatial:          5
K Celltype:         20
GAT Architecture:   4L × 512D × 4H
Dropout:        

In [2]:
st_file = f"/home/maweicheng/ST_data/GSE243275/GSM7782699_ST.h5ad"
output_dir = f"/home/maweicheng/ST_data/GSE243275/evaluate"   

In [3]:
import spagraph as spg

spg.cellcom(
    deconv_dir=output_dir,
    st_h5ad=st_file, 
    output_dir=output_dir,
    ligand_expr_threshold=3,
    receptor_expr_threshold=3,
    allow_same_celltype_comm=True,
    n_spot_neighbors=8,
    epochs=200,
    batch_size=64,
    seed=42,
    device= "cuda:1"
)


Stage 3: Cell Communication
Sample:             GSM7782699_ST
Device:             cuda:1
Epochs:             200
LR:                 0.0001
Batch Size:         64
Num Workers:        0
K Spot Neighbors:   8
Use HVG for Comm:   False
Ligand Expr Thr:    3 (CP10k)
Receptor Expr Thr:  3 (CP10k)
LR Score Thr:       1
Same-Type Comm:     True
Min Comm Edges:     1
Attention Thr:      1.0
MLP:               256,128 -> 64
GAT Architecture:   3L × [512, 256, 128]D × 8H
Dropout:            0.3
Loss Weights:
  Mask Recon:       1.0 (ratio=0.2)
  Node Recon:       0.5 (ratio=0.15)
Mask Seed:          1234
Early Stop:         patience=20, min_delta=0.1
Seed:               42
Output Dir:         /home/maweicheng/ST_data/GSE243275/evaluate

Stage 3.1: Load Data
LR pairs:           1942


ST shape:           (4992, 18085)
Spots:              4992
Clusters:           19

Stage 3.2: Load Spot-Cell Expression
Avg spot counts:    23096.5
Spot-cell expr:     /home/maweicheng/ST_data/GSE243275/evaluate/GSM7782699_ST_spot_cell_expr.csv
Spot-cell shape:    (34250, 18085)
Spot-cells kept:    34250

Stage 3.3: Select HVGs
HVG genes:          2000 / 18085
Cell types:         19
Composition shape:  (4992, 19)

Stage 3.5: Precompute LR Scores
LR scores:          spots=4992, knn=8, dist_thr=500.0, lig_thr=3, rec_thr=3, lr_thr=1
LR scores:          celltypes=19, spot-cells=34250
LR scores:          comm genes (ALL)
LR scores:          valid LR pairs = 1794/1942
LR scores:          events=8415660, neighbor_pairs=39936, spots_with_cells=4992/4992, same_type=ON
LR scores saved:    /home/maweicheng/ST_data/GSE243275/evaluate/lr_scores.csv (events=8415660, spot_pairs=39936, cell_pairs=1791161)
[Loaded] Spot pairs with communication: 39936
[Loaded] Spot-cell pairs with communication: 179116

Training:   0%|          | 0/200 [00:00<?, ?it/s]

Early stop:         patience=20, min_delta=0.1


Training:  28%|██▊       | 57/200 [30:15<1:15:54, 31.85s/it, Train=7.8914, Val=28.2242, Train_Mask=3.4391, Train_Node=8.9045]   


Early stop triggered: val loss not improved for 20 epochs (best=28.055725, current=31.500751)

Evaluating:          spots=4992, batches=78


Evaluating: 100%|██████████| 78/78 [05:20<00:00,  4.11s/it]


Eval batches used:   78
Edges collected:     8992593 (avg/spot=1801.4)
Post-process:        degree-scaled attention

Stage 3.6: Evaluate (attention importance)
Attention tensors:  4992
Edges total:        8992593 (unique_spots=4992)
Filter by attention: thr=1.0, kept=4030649/8992593 (44.8%)
LR mapping loaded:  508 pairs
LR scores used:     processed_edges=8992593, skipped_no_lr=0
LR stats saved:     /home/maweicheng/ST_data/GSE243275/evaluate/lr_pair_statistics.csv (unique_lr_pairs=330)

Top 10 LR pairs by occurrence:
  1. APP_CD74: n=4991, avg_attention=0.9942, spots=4991
  2. FN1_CD44: n=4830, avg_attention=1.4006, spots=4830
  3. COL1A1_CD44: n=4699, avg_attention=0.6275, spots=4699
  4. CXCL12_CXCR4: n=4411, avg_attention=1.3830, spots=4411
  5. SPP1_CD44: n=4305, avg_attention=0.6353, spots=4305
  6. MDK_LRP1: n=4264, avg_attention=0.9853, spots=4264
  7. CDH2_CDH2: n=4043, avg_attention=0.8041, spots=4043
  8. C3_C3AR1: n=3399, avg_attention=1.1539, spots=3399
  9. MDK_SDC1: n=30